# 07 - LSTM Forecasting Model

This notebook implements a Long Short-Term Memory (LSTM) deep learning model
for retail demand forecasting using sequential time-series learning.

Objectives:
- Prepare sequential forecasting data
- Train an LSTM forecasting model
- Evaluate deep learning forecasting performance
- Compare against baseline and ML models

Methodology Notes:
- LSTM uses a reduced large-scale subset of the engineered dataset
- Chronological train/test splitting is enforced
- Reproducible preprocessing pipeline is maintained

In [25]:
# ============================================
# Core Libraries
# ============================================

import warnings
warnings.filterwarnings("ignore")

import os
import random

import numpy as np
import pandas as pd

# ============================================
# Visualization
# ============================================

import matplotlib.pyplot as plt
import seaborn as sns

# ============================================
# Sklearn
# ============================================

from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

# ============================================
# TensorFlow / Keras
# ============================================

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

# ============================================
# Display Settings
# ============================================

pd.set_option("display.max_columns", None)

print("TensorFlow Version:", tf.__version__)

TensorFlow Version: 2.16.2


In [26]:
# ============================================
# Reproducibility
# ============================================

SEED = 42

np.random.seed(SEED)
random.seed(SEED)
tf.random.set_seed(SEED)

print("Random seeds initialized.")

Random seeds initialized.


In [27]:
# ============================================
# Create Output Directories
# ============================================

os.makedirs("../models", exist_ok=True)
os.makedirs("../artifacts", exist_ok=True)
os.makedirs("../figures", exist_ok=True)

print("Project directories ready.")

Project directories ready.


In [28]:
# ============================================
# Load Engineered Dataset
# ============================================

DATA_PATH = "../data/processed/feature_engineered_data.parquet"

df = pd.read_parquet(DATA_PATH)

print("Dataset loaded successfully.")

Dataset loaded successfully.


In [29]:
# ============================================
# Dataset Shape
# ============================================

print(f"Dataset Shape: {df.shape}")

Dataset Shape: (56650420, 40)


In [30]:
# ============================================
# Column Inspection
# ============================================

print(df.columns.tolist())

['id', 'item_id', 'dept_id', 'cat_id', 'store_id', 'state_id', 'day', 'sales', 'date', 'wm_yr_wk', 'weekday', 'wday', 'month', 'year', 'd', 'event_name_1', 'event_type_1', 'event_name_2', 'event_type_2', 'snap_CA', 'snap_TX', 'snap_WI', 'sell_price', 'rolling_mean_28', 'rolling_std_28', 'is_outlier', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_std_7', 'day_of_week', 'week_of_year', 'is_weekend', 'is_snap_day', 'is_sporting_event', 'is_cultural_event', 'is_national_event', 'is_religious_event', 'price_change']


In [ ]:
# ============================================
# Dataset Preview
# ============================================

df.head()

In [ ]:
# ============================================
# Data Types
# ============================================

df.dtypes

In [33]:
# ============================================
# Convert Date Column
# ============================================

df["date"] = pd.to_datetime(df["date"])

print(df["date"].dtype)

datetime64[ns]


In [34]:
# ============================================
# Missing Values
# ============================================

missing_values = df.isnull().sum()

missing_values[missing_values > 0].sort_values(ascending=False)

Series([], dtype: int64)

In [11]:
# ============================================
# Memory Usage
# ============================================

memory_usage_gb = df.memory_usage(deep=True).sum() / (1024**3)

print(f"Memory Usage: {memory_usage_gb:.2f} GB")

Memory Usage: 18.19 GB


In [35]:
# ============================================
# Unique Entity Analysis
# ============================================

n_stores = df["store_id"].nunique()
n_items = df["item_id"].nunique()

print("Unique Stores :", n_stores)
print("Unique Items  :", n_items)

Unique Stores : 10
Unique Items  : 3049


In [36]:
# ============================================
# Create Series Identifier
# ============================================

df["series_id"] = (
    df["store_id"].astype(str)
    + "_"
    + df["item_id"].astype(str)
)

print("Unique Series:", df["series_id"].nunique())

Unique Series: 30490


## Build LSTM Training Subset

In [16]:
# ============================================
# Chronological Sorting
# ============================================

df = df.sort_values(
    by=["series_id", "date"]
).reset_index(drop=True)

print("Dataset sorted chronologically.")

Dataset sorted chronologically.


In [37]:
# ============================================
# Series-Level Statistics
# ============================================

series_stats = (
    df.groupby("series_id")
      .agg(
          num_observations=("sales", "size"),
          mean_sales=("sales", "mean"),
          nonzero_ratio=("sales", lambda x: (x > 0).mean()),
          category=("cat_id", "first"),
          store=("store_id", "first")
      )
      .reset_index()
)

series_stats.head()

,series_id,num_observations,mean_sales,nonzero_ratio,category,store
0,0_0,1858,0.767492,0.429494,0,0
1,0_1,1858,0.480624,0.332616,0,0
2,0_10,1858,2.805167,0.684607,0,0
3,0_100,1858,1.368676,0.549516,0,0
4,0_1000,1858,1.595802,0.586114,0,0


In [38]:
MIN_OBSERVATIONS = 365
MIN_NONZERO_RATIO = 0.30

# ============================================
# Filter Weak Series
# ============================================

filtered_series = series_stats[
    (series_stats["num_observations"] >= MIN_OBSERVATIONS)
    &
    (series_stats["nonzero_ratio"] >= MIN_NONZERO_RATIO)
]

print("Remaining Series:", len(filtered_series))

Remaining Series: 13815


In [39]:
SERIES_PER_CATEGORY = 400

# ============================================
# Stratified Sampling
# ============================================

sampled_series = (
    filtered_series
    .groupby("category", group_keys=False)
    .apply(
        lambda x: x.sample(
            n=min(SERIES_PER_CATEGORY, len(x)),
            random_state=SEED
        )
    )
)

print("Sampled Series:", len(sampled_series))

Sampled Series: 1200


In [40]:
# ============================================
# Build LSTM Dataset
# ============================================

selected_series_ids = sampled_series["series_id"].unique()

lstm_df = df[
    df["series_id"].isin(selected_series_ids)
].copy()

print("LSTM Dataset Shape:", lstm_df.shape)

LSTM Dataset Shape: (2229600, 41)


In [41]:
# ============================================
# Category Distribution
# ============================================

lstm_df["cat_id"].value_counts()

cat_id
0    743200
1    743200
2    743200
Name: count, dtype: int64

In [22]:
# ============================================
# Memory Usage
# ============================================

memory_usage_gb = (
    lstm_df.memory_usage(deep=True).sum()
    / (1024**3)
)

print(f"LSTM Dataset Memory Usage: {memory_usage_gb:.2f} GB")

LSTM Dataset Memory Usage: 0.76 GB


In [42]:
# ============================================
# Save Sampling Metadata
# ============================================

filtered_series.to_parquet(
    "../artifacts/lstm_filtered_series.parquet",
    index=False
)

sampled_series.to_parquet(
    "../artifacts/lstm_sampled_series.parquet",
    index=False
)

print("Sampling metadata saved.")

Sampling metadata saved.


Define Final Feature Set

In [45]:
# ============================================
# Final LSTM Feature Set
# ============================================

FEATURE_COLUMNS = [
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_28",
    "rolling_std_28",
    "sell_price",
    "price_change",
    "month",
    "wday",
    "is_weekend",
    "is_snap_day",
    "is_sporting_event",
    "is_cultural_event",
    "is_national_event",
    "is_religious_event"
]

TARGET_COLUMN = "sales"

print("Number of Features:", len(FEATURE_COLUMNS))

Number of Features: 17


In [52]:
# ============================================
# Rebuild Modeling Dataset
# ============================================

model_df = lstm_df[
    ["series_id", "date"] + FEATURE_COLUMNS + [TARGET_COLUMN]
].copy()

print(model_df.shape)
print(model_df.columns.tolist())

(2229600, 20)
['series_id', 'date', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28', 'sell_price', 'price_change', 'month', 'wday', 'is_weekend', 'is_snap_day', 'is_sporting_event', 'is_cultural_event', 'is_national_event', 'is_religious_event', 'sales']


In [53]:
# ============================================
# Remove Missing Values
# ============================================

model_df = model_df.dropna().reset_index(drop=True)

print("Shape After NA Removal:", model_df.shape)

Shape After NA Removal: (2229600, 20)


In [44]:
# ============================================
# Proper Chronological Split
# ============================================

train_df = model_df[
    model_df["date"] <= "2015-12-31"
].copy()

validation_df = model_df[
    (model_df["date"] >= "2016-01-01") &
    (model_df["date"] <= "2016-03-31")
].copy()

test_df = model_df[
    model_df["date"] >= "2016-04-01"
].copy()

print("Train Shape      :", train_df.shape)
print("Validation Shape :", validation_df.shape)
print("Test Shape       :", test_df.shape)

Train Shape      : (2091600, 20)
Validation Shape : (109200, 20)
Test Shape       : (28800, 20)


In [45]:
# ============================================
# Feature / Target Separation
# ============================================

X_train_df = train_df[FEATURE_COLUMNS].copy()
X_validation_df = validation_df[FEATURE_COLUMNS].copy()
X_test_df = test_df[FEATURE_COLUMNS].copy()

y_train_df = train_df[[TARGET_COLUMN]].copy()
y_validation_df = validation_df[[TARGET_COLUMN]].copy()
y_test_df = test_df[[TARGET_COLUMN]].copy()

print(X_train_df.shape)
print(X_validation_df.shape)
print(X_test_df.shape)

(2091600, 17)
(109200, 17)
(28800, 17)


In [46]:
# ============================================
# Scaling
# ============================================

from sklearn.preprocessing import MinMaxScaler

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

X_train_scaled = feature_scaler.fit_transform(X_train_df)

X_validation_scaled = feature_scaler.transform(X_validation_df)
X_test_scaled = feature_scaler.transform(X_test_df)

y_train_scaled = target_scaler.fit_transform(y_train_df)

y_validation_scaled = target_scaler.transform(y_validation_df)
y_test_scaled = target_scaler.transform(y_test_df)

print("Scaling complete.")

Scaling complete.


In [47]:
# ============================================
# Rebuild Scaled Train DataFrame
# ============================================

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=FEATURE_COLUMNS,
    index=train_df.index
)

y_train_scaled_df = pd.DataFrame(
    y_train_scaled,
    columns=[TARGET_COLUMN],
    index=train_df.index
)

train_scaled_df = pd.concat(
    [
        train_df[["series_id", "date"]].reset_index(drop=True),
        X_train_scaled_df.reset_index(drop=True),
        y_train_scaled_df.reset_index(drop=True)
    ],
    axis=1
)

print(train_scaled_df.shape)

(2091600, 20)


In [48]:
# ============================================
# Rebuild Scaled Validation DataFrame
# ============================================

X_validation_scaled_df = pd.DataFrame(
    X_validation_scaled,
    columns=FEATURE_COLUMNS,
    index=validation_df.index
)

y_validation_scaled_df = pd.DataFrame(
    y_validation_scaled,
    columns=[TARGET_COLUMN],
    index=validation_df.index
)

validation_scaled_df = pd.concat(
    [
        validation_df[["series_id", "date"]].reset_index(drop=True),
        X_validation_scaled_df.reset_index(drop=True),
        y_validation_scaled_df.reset_index(drop=True)
    ],
    axis=1
)

print(validation_scaled_df.shape)

(109200, 20)


In [49]:
# ============================================
# Rebuild Scaled Test DataFrame
# ============================================

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=FEATURE_COLUMNS,
    index=test_df.index
)

y_test_scaled_df = pd.DataFrame(
    y_test_scaled,
    columns=[TARGET_COLUMN],
    index=test_df.index
)

test_scaled_df = pd.concat(
    [
        test_df[["series_id", "date"]].reset_index(drop=True),
        X_test_scaled_df.reset_index(drop=True),
        y_test_scaled_df.reset_index(drop=True)
    ],
    axis=1
)

print(test_scaled_df.shape)

(28800, 20)


In [50]:
# ============================================
# Sequence Length
# ============================================

SEQUENCE_LENGTH = 28

print("Sequence Length:", SEQUENCE_LENGTH)

Sequence Length: 28


In [51]:
# ============================================
# Sequence Generation Function
# ============================================

def create_sequences(data, feature_cols, target_col, sequence_length):

    X = []
    y = []

    features = data[feature_cols].values
    target = data[target_col].values

    for i in range(sequence_length, len(data)):

        X.append(
            features[i-sequence_length:i]
        )

        y.append(
            target[i]
        )

    return np.array(X), np.array(y)

In [52]:
# ============================================
# Generate Train Sequences
# ============================================

X_train_sequences = []
y_train_sequences = []

train_groups = train_scaled_df.groupby("series_id")

for _, group in train_groups:

    group = group.sort_values("date")

    X_seq, y_seq = create_sequences(
        data=group,
        feature_cols=FEATURE_COLUMNS,
        target_col=TARGET_COLUMN,
        sequence_length=SEQUENCE_LENGTH
    )

    if len(X_seq) > 0:

        X_train_sequences.append(X_seq)
        y_train_sequences.append(y_seq)

print("Train sequence generation complete.")

Train sequence generation complete.


In [53]:
# ============================================
# Concatenate Train Tensors
# ============================================

X_train = np.concatenate(X_train_sequences)

y_train = np.concatenate(y_train_sequences)

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (2058000, 28, 17)
y_train shape: (2058000,)


In [54]:
print(f"X_train Memory: {X_train.nbytes / (1024**3):.2f} GB")

X_train Memory: 7.30 GB


In [55]:
# ============================================
# Save Train Tensors
# ============================================

np.save("../artifacts/X_train.npy", X_train)
np.save("../artifacts/y_train.npy", y_train)

print("Train tensors saved.")

Train tensors saved.


In [56]:
# ============================================
# Free Memory
# ============================================

del X_train_sequences
del y_train_sequences

import gc
gc.collect()

print("Memory cleared.")

Memory cleared.


In [57]:
# ============================================
# Generate Validation Sequences
# ============================================

X_validation_sequences = []
y_validation_sequences = []

validation_groups = validation_scaled_df.groupby("series_id")

for _, group in validation_groups:

    group = group.sort_values("date")

    X_seq, y_seq = create_sequences(
        data=group,
        feature_cols=FEATURE_COLUMNS,
        target_col=TARGET_COLUMN,
        sequence_length=SEQUENCE_LENGTH
    )

    if len(X_seq) > 0:

        X_validation_sequences.append(X_seq)
        y_validation_sequences.append(y_seq)

print("Validation sequence generation complete.")

Validation sequence generation complete.


In [58]:
# ============================================
# Concatenate Validation Tensors
# ============================================

X_validation = np.concatenate(X_validation_sequences)

y_validation = np.concatenate(y_validation_sequences)

print("X_validation shape:", X_validation.shape)
print("y_validation shape:", y_validation.shape)

X_validation shape: (75600, 28, 17)
y_validation shape: (75600,)


In [59]:
# ============================================
# Validation Tensor Memory
# ============================================

print(
    f"X_validation Memory: "
    f"{X_validation.nbytes / (1024**3):.2f} GB"
)

X_validation Memory: 0.27 GB


In [60]:
# ============================================
# Save Validation Tensors
# ============================================

np.save("../artifacts/X_validation.npy", X_validation)
np.save("../artifacts/y_validation.npy", y_validation)

print("Validation tensors saved.")

Validation tensors saved.


In [61]:
# ============================================
# Free Validation Memory
# ============================================

del X_validation_sequences
del y_validation_sequences

gc.collect()

print("Validation memory cleared.")

Validation memory cleared.


In [62]:
# ============================================
# Generate Test Sequences
# ============================================

X_test_sequences = []
y_test_sequences = []

test_groups = test_scaled_df.groupby("series_id")

for _, group in test_groups:

    group = group.sort_values("date")

    X_seq, y_seq = create_sequences(
        data=group,
        feature_cols=FEATURE_COLUMNS,
        target_col=TARGET_COLUMN,
        sequence_length=SEQUENCE_LENGTH
    )

    if len(X_seq) > 0:

        X_test_sequences.append(X_seq)
        y_test_sequences.append(y_seq)

print("Test sequence generation complete.")

Test sequence generation complete.


In [63]:
# ============================================
# Concatenate Test Tensors
# ============================================

X_test = np.concatenate(X_test_sequences)

y_test = np.concatenate(y_test_sequences)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

ValueError: need at least one array to concatenate

In [56]:
# ============================================
# Initialize Scalers
# ============================================

from sklearn.preprocessing import MinMaxScaler

feature_scaler = MinMaxScaler()
target_scaler = MinMaxScaler()

In [57]:
# ============================================
# Fit Scalers on Training Data ONLY
# ============================================

X_train_scaled = feature_scaler.fit_transform(X_train_df)

X_test_scaled = feature_scaler.transform(X_test_df)

y_train_scaled = target_scaler.fit_transform(y_train_df)

y_test_scaled = target_scaler.transform(y_test_df)

print("Scaling complete.")

Scaling complete.


In [58]:
print(X_train_scaled.shape)
print(X_test_scaled.shape)

print(y_train_scaled.shape)
print(y_test_scaled.shape)

(1870800, 17)
(358800, 17)
(1870800, 1)
(358800, 1)


In [59]:
# ============================================
# Scaled Feature DataFrames
# ============================================

X_train_scaled_df = pd.DataFrame(
    X_train_scaled,
    columns=FEATURE_COLUMNS,
    index=train_df.index
)

X_test_scaled_df = pd.DataFrame(
    X_test_scaled,
    columns=FEATURE_COLUMNS,
    index=test_df.index
)

In [60]:
# ============================================
# Scaled Target DataFrames
# ============================================

y_train_scaled_df = pd.DataFrame(
    y_train_scaled,
    columns=[TARGET_COLUMN],
    index=train_df.index
)

y_test_scaled_df = pd.DataFrame(
    y_test_scaled,
    columns=[TARGET_COLUMN],
    index=test_df.index
)

In [61]:
# ============================================
# Rebuild Scaled Train Dataset
# ============================================

train_scaled_df = pd.concat(
    [
        train_df[["series_id", "date"]].reset_index(drop=True),
        X_train_scaled_df.reset_index(drop=True),
        y_train_scaled_df.reset_index(drop=True)
    ],
    axis=1
)

print(train_scaled_df.shape)
train_scaled_df.head()

(1870800, 20)


,series_id,date,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_28,rolling_std_28,sell_price,price_change,month,wday,is_weekend,is_snap_day,is_sporting_event,is_cultural_event,is_national_event,is_religious_event,sales
0,0_1020,2011-03-25,0.000000,0.000000,0.000000,0.010883,0.023893,0.009918,0.034002,0.141605,0.458689,0.181818,1.000000,0.0,0.0,0.0,0.0,1.0,0.0,0.000000
1,0_1020,2011-03-26,0.000000,0.003650,0.003401,0.009674,0.024466,0.009918,0.034251,0.141605,0.458689,0.181818,0.000000,1.0,0.0,0.0,0.0,1.0,0.0,0.000000
2,0_1020,2011-03-27,0.007299,0.021898,0.027211,0.004837,0.010528,0.013368,0.024376,0.141605,0.458689,0.181818,0.166667,1.0,0.0,0.0,0.0,1.0,0.0,0.000000
3,0_1020,2011-03-28,0.003650,0.000000,0.000000,0.006046,0.010262,0.012937,0.024447,0.141605,0.458689,0.181818,0.333333,0.0,0.0,0.0,0.0,1.0,0.0,0.007299
4,0_1020,2011-03-29,0.007299,0.000000,0.000000,0.008464,0.010788,0.012937,0.024205,0.141605,0.458689,0.181818,0.500000,0.0,0.0,0.0,0.0,1.0,0.0,0.003650


In [62]:
# ============================================
# Rebuild Scaled Test Dataset
# ============================================

test_scaled_df = pd.concat(
    [
        test_df[["series_id", "date"]].reset_index(drop=True),
        X_test_scaled_df.reset_index(drop=True),
        y_test_scaled_df.reset_index(drop=True)
    ],
    axis=1
)

print(test_scaled_df.shape)
test_scaled_df.head()

(358800, 20)


,series_id,date,lag_7,lag_14,lag_28,rolling_mean_7,rolling_std_7,rolling_mean_28,rolling_std_28,sell_price,price_change,month,wday,is_weekend,is_snap_day,is_sporting_event,is_cultural_event,is_national_event,is_religious_event,sales
0,0_1020,2015-07-01,0.0,0.00000,0.000000,0.004837,0.008488,0.006468,0.021778,0.132841,0.458689,0.545455,0.666667,0.0,1.0,0.0,0.0,1.0,0.0,0.00000
1,0_1020,2015-07-02,0.0,0.00000,0.000000,0.004837,0.008488,0.006468,0.021728,0.132841,0.458689,0.545455,0.833333,0.0,1.0,0.0,0.0,1.0,0.0,0.00365
2,0_1020,2015-07-03,0.0,0.00000,0.013605,0.004837,0.008488,0.008193,0.018142,0.132841,0.458689,0.545455,1.000000,0.0,1.0,0.0,0.0,1.0,0.0,0.00000
3,0_1020,2015-07-04,0.0,0.00365,0.013605,0.003628,0.008488,0.009918,0.013122,0.132841,0.458689,0.545455,0.000000,1.0,1.0,1.0,0.0,0.0,0.0,0.00000
4,0_1020,2015-07-05,0.0,0.00000,0.000000,0.003628,0.008488,0.009056,0.013122,0.132841,0.458689,0.545455,0.166667,1.0,1.0,0.0,0.0,1.0,0.0,0.00000


In [63]:
print(train_scaled_df.columns.tolist())

['series_id', 'date', 'lag_7', 'lag_14', 'lag_28', 'rolling_mean_7', 'rolling_std_7', 'rolling_mean_28', 'rolling_std_28', 'sell_price', 'price_change', 'month', 'wday', 'is_weekend', 'is_snap_day', 'is_sporting_event', 'is_cultural_event', 'is_national_event', 'is_religious_event', 'sales']


In [64]:
# ============================================
# Sequence Length
# ============================================

SEQUENCE_LENGTH = 28

print("Sequence Length:", SEQUENCE_LENGTH)

Sequence Length: 28


In [65]:
# ============================================
# Sequence Generation Function
# ============================================

def create_sequences(data, feature_cols, target_col, sequence_length):
    
    X = []
    y = []
    
    features = data[feature_cols].values
    target = data[target_col].values
    
    for i in range(sequence_length, len(data)):
        
        X.append(
            features[i-sequence_length:i]
        )
        
        y.append(
            target[i]
        )
    
    return np.array(X), np.array(y)

In [75]:
# ============================================
# Generate Test Sequences
# ============================================

X_test_sequences = []
y_test_sequences = []

test_groups = test_scaled_df.groupby("series_id")

for _, group in test_groups:

    group = group.sort_values("date")

    X_seq, y_seq = create_sequences(
        data=group,
        feature_cols=FEATURE_COLUMNS,
        target_col=TARGET_COLUMN,
        sequence_length=SEQUENCE_LENGTH
    )

    if len(X_seq) > 0:

        X_test_sequences.append(X_seq)
        y_test_sequences.append(y_seq)

print("Test sequence generation complete.")

Test sequence generation complete.


In [ ]:
# ============================================
# Save Test Tensors
# ============================================

np.save("../artifacts/X_test.npy", X_test)
np.save("../artifacts/y_test.npy", y_test)

print("Test tensors saved.")

In [76]:
# ============================================
# Concatenate Test Tensors
# ============================================

X_test = np.concatenate(X_test_sequences)

y_test = np.concatenate(y_test_sequences)

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (325200, 28, 17)
y_test shape: (325200,)


In [77]:
# ============================================
# Save Test Tensors
# ============================================

np.save("../artifacts/X_test.npy", X_test)
np.save("../artifacts/y_test.npy", y_test)

print("Test tensors saved.")

Test tensors saved.


In [78]:
x_test_size = os.path.getsize("../artifacts/X_test.npy") / (1024**3)

print(f"X_test Size: {x_test_size:.2f} GB")

X_test Size: 1.15 GB


In [1]:
# ============================================
# Imports
# ============================================

import numpy as np
import pandas as pd

import tensorflow as tf

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (
    LSTM,
    Dense,
    Dropout
)

from tensorflow.keras.callbacks import EarlyStopping

from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error
)

print("Libraries loaded.")

Libraries loaded.


In [2]:
# ============================================
# Load Training Tensors
# ============================================

X_train = np.load("../artifacts/X_train.npy")
y_train = np.load("../artifacts/y_train.npy")

print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_train shape: (1837200, 28, 17)
y_train shape: (1837200,)


In [3]:
# ============================================
# Load Test Tensors
# ============================================

X_test = np.load("../artifacts/X_test.npy")
y_test = np.load("../artifacts/y_test.npy")

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)

X_test shape: (325200, 28, 17)
y_test shape: (325200,)


In [4]:
# ============================================
# Tensor Memory Inspection
# ============================================

print(f"X_train Memory: {X_train.nbytes / (1024**3):.2f} GB")
print(f"X_test Memory : {X_test.nbytes / (1024**3):.2f} GB")

X_train Memory: 6.52 GB
X_test Memory : 1.15 GB


In [5]:
# ============================================
# Build LSTM Model
# ============================================

model = Sequential([

    LSTM(
        64,
        input_shape=(X_train.shape[1], X_train.shape[2]),
        return_sequences=True
    ),

    Dropout(0.2),

    LSTM(32),

    Dropout(0.2),

    Dense(1)
])

model.summary()

2026-05-17 20:40:20.141648: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M3
2026-05-17 20:40:20.141872: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 24.00 GB
2026-05-17 20:40:20.141882: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 8.88 GB
2026-05-17 20:40:20.142106: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2026-05-17 20:40:20.142119: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)
/Users/desmond/Capstone Project/retail-demand-forecasting/retail_forecasting_env/lib/python3.11/site-packages/keras/src/layers/rnn/rnn.py:199: UserWarning: Do not pass an `input_shape`/`

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm (LSTM)                     │ (None, 28, 64)         │        20,992 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 28, 64)         │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_1 (LSTM)                   │ (None, 32)             │        12,416 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 32)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 33,441 (130.63 KB)

 Trainable params: 33,441 (130.63 KB)

 Non-trainable params: 0 (0.00 B)

In [6]:
# ============================================
# Compile Model
# ============================================

model.compile(
    optimizer="adam",
    loss="mse",
    metrics=["mae"]
)

print("Model compiled.")

Model compiled.


In [7]:
# ============================================
# Early Stopping
# ============================================

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

In [8]:
# ============================================
# Training Configuration
# ============================================

BATCH_SIZE = 128
EPOCHS = 20

# ============================================
# Train Model
# ============================================

history = model.fit(
    X_train,
    y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop],
    verbose=1
)

Epoch 1/20


2026-05-17 20:59:20.775153: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:117] Plugin optimizer for device_type GPU is enabled.


14354/14354 ━━━━━━━━━━━━━━━━━━━━ 215s 15ms/step - loss: 1.3758e-04 - mae: 0.0060 - val_loss: 1.0585e-04 - val_mae: 0.0054
Epoch 2/20
14354/14354 ━━━━━━━━━━━━━━━━━━━━ 208s 14ms/step - loss: 1.2056e-04 - mae: 0.0054 - val_loss: 1.0269e-04 - val_mae: 0.0053
Epoch 3/20
14354/14354 ━━━━━━━━━━━━━━━━━━━━ 210s 15ms/step - loss: 1.1874e-04 - mae: 0.0053 - val_loss: 1.0429e-04 - val_mae: 0.0059
Epoch 4/20
14354/14354 ━━━━━━━━━━━━━━━━━━━━ 210s 15ms/step - loss: 1.1807e-04 - mae: 0.0053 - val_loss: 1.0274e-04 - val_mae: 0.0057
Epoch 5/20
14354/14354 ━━━━━━━━━━━━━━━━━━━━ 210s 15ms/step - loss: 1.1738e-04 - mae: 0.0053 - val_loss: 1.0106e-04 - val_mae: 0.0054
Epoch 6/20
14354/14354 ━━━━━━━━━━━━━━━━━━━━ 211s 15ms/step - loss: 1.1707e-04 - mae: 0.0053 - val_loss: 1.0213e-04 - val_mae: 0.0056
Epoch 7/20
14354/14354 ━━━━━━━━━━━━━━━━━━━━ 211s 15ms/step - loss: 1.1668e-04 - mae: 0.0053 - val_loss: 1.0397e-04 - val_mae: 0.0052
Epoch 8/20
14354/14354 ━━━━━━━━━━━━━━━━━━━━ 212s 15ms/step - loss: 1.1602e-04 - 

In [11]:
# ============================================
# Reload Dataset
# ============================================

df = pd.read_parquet("../artifacts/lstm_subset.parquet")

print(df.shape)

(2229600, 41)


In [12]:
# ============================================
# Feature Definitions
# ============================================

FEATURE_COLUMNS = [
    "lag_7",
    "lag_14",
    "lag_28",
    "rolling_mean_7",
    "rolling_std_7",
    "rolling_mean_28",
    "rolling_std_28",
    "sell_price",
    "price_change",
    "month",
    "wday",
    "is_weekend",
    "is_snap_day",
    "is_sporting_event",
    "is_cultural_event",
    "is_national_event",
    "is_religious_event"
]

TARGET_COLUMN = "sales"

In [13]:
# ============================================
# Modeling Dataset
# ============================================

model_df = df[
    ["series_id", "date"] + FEATURE_COLUMNS + [TARGET_COLUMN]
].copy()

model_df = model_df.dropna().reset_index(drop=True)

In [14]:
# ============================================
# Train/Test Split
# ============================================

TRAIN_END_DATE = "2015-06-30"

train_df = model_df[
    model_df["date"] <= TRAIN_END_DATE
].copy()

test_df = model_df[
    model_df["date"] > TRAIN_END_DATE
].copy()

In [15]:
# ============================================
# Rebuild Target Scaler
# ============================================

from sklearn.preprocessing import MinMaxScaler

target_scaler = MinMaxScaler()

target_scaler.fit(
    train_df[[TARGET_COLUMN]]
)

print("Target scaler rebuilt.")

Target scaler rebuilt.


In [17]:
# ============================================
# Generate Predictions
# ============================================

y_pred_scaled = model.predict(X_test)

print(y_pred_scaled.shape)

10163/10163 ━━━━━━━━━━━━━━━━━━━━ 34s 3ms/step
(325200, 1)


In [18]:
# ============================================
# Inverse Transform Predictions
# ============================================

y_pred = target_scaler.inverse_transform(
    y_pred_scaled
)

y_test_actual = target_scaler.inverse_transform(
    y_test.reshape(-1, 1)
)

print(y_pred.shape)
print(y_test_actual.shape)

(325200, 1)
(325200, 1)


In [19]:
# ============================================
# Evaluation Metrics
# ============================================

rmse = np.sqrt(
    mean_squared_error(
        y_test_actual,
        y_pred
    )
)

mae = mean_absolute_error(
    y_test_actual,
    y_pred
)

print(f"RMSE: {rmse:.4f}")
print(f"MAE : {mae:.4f}")

RMSE: 2.7544
MAE : 1.4662


In [20]:
# ============================================
# Save Trained Model
# ============================================

model.save("../models/lstm_model.keras")

print("LSTM model saved.")

LSTM model saved.


In [21]:
# ============================================
# Save Predictions
# ============================================

predictions_df = pd.DataFrame({
    "actual_sales": y_test_actual.flatten(),
    "predicted_sales": y_pred.flatten()
})

predictions_df.to_parquet(
    "../artifacts/lstm_predictions.parquet",
    index=False
)

print("Predictions saved.")

Predictions saved.


In [22]:
# ============================================
# Save Metrics
# ============================================

metrics_df = pd.DataFrame({
    "model": ["LSTM"],
    "RMSE": [rmse],
    "MAE": [mae]
})

metrics_df.to_csv(
    "../artifacts/lstm_metrics.csv",
    index=False
)

metrics_df

,model,RMSE,MAE
0,LSTM,2.754437,1.466168
